## Extracting Text Preserving Layout

In [ ]:
from docling.document_converter import DocumentConverter

source = (
    "2403.09724_cropped.pdf"  # "https://arxiv.org/pdf/2408.09869"  # PDF path or URL
)
converter = DocumentConverter()
result = converter.convert_single(source)

In [ ]:
print(result.render_as_markdown())  # output: "## Docling Technical Report[...]"

In [ ]:
print(result.render_as_doctags())  # output: "<document><title><page_1><loc_20>..."

In [33]:
import json

# Convert the result to a dictionary
result_dict = result.render_as_dict()

# Define the output file path
output_file_path = "5k5tlOpUAq.json"

# Save the dictionary to a JSON file
with open(output_file_path, "w") as json_file:
    json.dump(result_dict, json_file, indent=4)

In [ ]:
import pandas as pd

pd.set_option("display.max_rows", None)

pd.DataFrame(result.render_as_dict().get("main-text")).query("type == 'subtitle-level-1'")

In [ ]:
print(result.render_as_dict())

In [ ]:
result.render_as_text()

In [53]:
import json
import markdown_to_json

jsonified = markdown_to_json.jsonify(result.render_as_markdown())

with open("2403.09724.json", "w") as json_file:
    json.dump(json.loads(jsonified), json_file, indent=4)

In [ ]:
jsonified

## Extracting Figures from PDF

In [ ]:
from docling.datamodel.base_models import (
    AssembleOptions,
    ConversionStatus,
    FigureElement,
    PageElement,
    TableElement,
)

from pathlib import Path

IMAGE_RESOLUTION_SCALE = 2.0

# Important: For operating with page images, we must keep them, otherwise the DocumentConverter
# will destroy them for cleaning up memory.
# This is done by setting AssembleOptions.images_scale, which also defines the scale of images.
# scale=1 correspond of a standard 72 DPI image
assemble_options = AssembleOptions()
assemble_options.images_scale = IMAGE_RESOLUTION_SCALE

doc_converter = DocumentConverter(assemble_options=assemble_options)

conv_results = doc_converter.convert_single(source)
doc_filename = conv_results.input.file.stem

output_dir = Path("./images")
output_dir.mkdir(parents=True, exist_ok=True)

# Export figures and tables
for element, image in conv_results.render_element_images(
    element_types=(FigureElement, TableElement)
):
    element_image_filename = (
        output_dir / f"{doc_filename}-element-{element.id}.png"
    )
    with element_image_filename.open("wb") as fp:
        image.save(fp, "PNG")

## Extracting Tables

In [ ]:
import pandas as pd

output_dir_tab = Path("./tables")
output_dir_tab.mkdir(parents=True, exist_ok=True)

doc_filename = conv_results.input.file.stem

# Export tables
for table_ix, table in enumerate(conv_results.output.tables):
    table_df: pd.DataFrame = table.export_to_dataframe()
    # print(f"## Table {table_ix}")
    # print(table_df.to_markdown())

    # Save the table as csv
    element_csv_filename = output_dir_tab / f"{doc_filename}-table-{table_ix+1}.csv"

    table_df.to_csv(element_csv_filename)

    # Save the table as html
    element_html_filename = output_dir_tab / f"{doc_filename}-table-{table_ix+1}.html"

    with element_html_filename.open("w") as fp:
        fp.write(table.export_to_html())

## Chunking

In [ ]:
from docling.document_converter import DocumentConverter
from docling_core.transforms.chunker import HierarchicalChunker

doc = DocumentConverter().convert_single("1-s2.0-S1877050924011402-main.pdf").output
chunks = list(HierarchicalChunker(include_metadata=True, min_chunk_len=100).chunk(doc))

In [50]:
import fitz

def crop_pdf(input_pdf, output_pdf, left_margin, right_margin):
    document = fitz.open(input_pdf)
    for page in document:
        rect = page.rect  # Get the rectangle of the page
        new_rect = fitz.Rect(
            rect.x0 + left_margin, rect.y0, rect.x1 - right_margin, rect.y1
        )
        page.set_cropbox(new_rect)  # Set the new crop box

    document.save(output_pdf)

crop_pdf("2403.09724v4.pdf", "2403.09724_cropped.pdf", 35, 35)